# Mitochondrial Variant Calling

singlify calls mitochondrial DNA variants (heteroplasmy) from scRNA-seq data.
MT variants are valuable for:
- **Lineage tracing** — cells sharing MT mutations share a clonal origin
- **Sample QC** — abnormal MT mutation rates can indicate sample issues
- **Population genetics** — MT haplogroup inference from scRNA-seq

**Sample**: GSM3573650 (10x v3 PBMC, 75K cells)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

SAMPLE = '/mnt/projects/debruinz_project/singlify_pipeline/quant/scrna/GSE125/GSE125416/GSM3573650'

# Load MT variants
mt = pd.read_csv(Path(SAMPLE) / 'mt_variants.tsv', sep='\t')
print(f'Mitochondrial variants detected: {len(mt):,}')
print(f'MT genome positions covered: {mt["pos"].nunique():,} / 16,569')
print(f'\nTop 10 variants by # cells covered:')
print(mt.nlargest(10, 'n_cells_covered')[['pos', 'ref', 'alt', 'n_cells_covered', 'n_cells_het', 'mean_vaf']].to_string(index=False))

Mitochondrial variants detected: 3,717
MT genome positions covered: 3,717 / 16,569

Top 10 variants by # cells covered:
 pos ref alt  n_cells_covered  n_cells_het  mean_vaf
9806   C   G             6646          575  0.022074
9796   T   G             6645          776  0.023957
9807   A   C             6645          762  0.021749
9805   C   G             6643          491  0.023501
9797   T   G             6638          845  0.023634
9794   A   C             6637         1131  0.023262
9795   T   G             6635          674  0.022613
9801   G   T             6635         1016  0.023275
9804   G   T             6635         1155  0.023991
9798   T   G             6634          864  0.022493


In [2]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# Coverage along MT genome
axes[0,0].scatter(mt['pos'], mt['n_cells_covered'], alpha=0.3, s=5, color='#3b82f6')
axes[0,0].set_xlabel('MT Position')
axes[0,0].set_ylabel('Cells Covered')
axes[0,0].set_title('Variant Coverage Along MT Genome')
axes[0,0].set_xlim(0, 16569)

# VAF distribution
axes[0,1].hist(mt['mean_vaf'], bins=50, color='#22c55e', edgecolor='white')
axes[0,1].set_xlabel('Mean Variant Allele Frequency')
axes[0,1].set_ylabel('Variants')
axes[0,1].set_title('VAF Distribution')
axes[0,1].axvline(0.5, color='red', linestyle='--', alpha=0.5, label='Heteroplasmic peak')
axes[0,1].legend()

# Cells per variant
axes[1,0].hist(np.log10(mt['n_cells_het']+1), bins=40, color='#f59e0b', edgecolor='white')
axes[1,0].set_xlabel('log10(Heteroplasmic Cells + 1)')
axes[1,0].set_ylabel('Variants')
axes[1,0].set_title('Heteroplasmic Cell Count per Variant')

# Mutation spectrum
mt['mutation'] = mt['ref'] + '>' + mt['alt']
spec = mt['mutation'].value_counts().head(12)
axes[1,1].barh(spec.index, spec.values, color='#8b5cf6')
axes[1,1].set_xlabel('Count')
axes[1,1].set_title('Mutation Spectrum')

plt.suptitle('MT Heteroplasmy — GSM3573650 (3,717 variants)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('mt_variants.png', dpi=150, bbox_inches='tight')
plt.show()

In [3]:
# High-confidence clonal variants (many cells, moderate VAF)
clonal = mt[(mt['n_cells_het'] >= 50) & (mt['mean_vaf'] > 0.1) & (mt['mean_vaf'] < 0.9)]
print(f'\n═══ High-Confidence Clonal Variants ═══')
print(f'Criteria: ≥50 het cells, 0.1 < VAF < 0.9')
print(f'Found: {len(clonal)} variants\n')
if len(clonal) > 0:
    print(clonal[['pos', 'ref', 'alt', 'n_cells_covered', 'n_cells_het', 'mean_vaf']]
          .sort_values('n_cells_het', ascending=False).head(20).to_string(index=False))


═══ High-Confidence Clonal Variants ═══
Criteria: ≥50 het cells, 0.1 < VAF < 0.9
Found: 1 variants

 pos ref alt  n_cells_covered  n_cells_het  mean_vaf
2617   A   T               61           61  0.376356


## Applications

1. **Clonal lineage tracing**: Shared MT mutations indicate cells from the same clonal expansion
2. **Donor deconvolution**: When genetic demux fails, MT variants can distinguish donors
3. **QC flag**: Unusually high MT mutation rates may indicate poor sample quality
4. **Tumor heterogeneity**: In cancer samples, MT heteroplasmy correlates with subclonal structure

singlify outputs both:
- `mt_variants.tsv` — per-position summary (loaded in this notebook)
- `mt_heteroplasmy.1pz` — full cells × positions sparse matrix (loadable via `load_dir(layer='mt_heteroplasmy')`)